# 4장 실습: Transformer 아키텍처 밑바닥 구현

이 노트북에서는 Transformer의 핵심 구성 요소를 하나씩 밑바닥부터 구현합니다.

| 순서 | 내용 | ch4A.md 섹션 |
|------|------|-------------|
| 1 | Positional Encoding (Sinusoidal / Learned) | 4.3 |
| 2 | Residual Connection & Layer Normalization | 4.4 |
| 3 | Multi-Head Attention + FFN = Encoder Block | 4.4 |
| 4 | Decoder Block (Causal Mask + Cross-Attention) | 4.5 |
| 5 | 전체 Transformer Encoder 스택 + 분류 모델 | 4.4–4.5 |

In [1]:
# ============================================================
# 환경 확인 (루트의 setup_env.py로 이미 설치됨)
# ============================================================
import torch
import numpy as np
import matplotlib

print(f"PyTorch    {torch.__version__}")
print(f"NumPy      {np.__version__}")
print(f"Matplotlib {matplotlib.__version__}")

# GPU 확인
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print("Apple Silicon GPU (MPS) 사용 가능")
else:
    print("CPU 모드 (실습에는 문제없음)")

print("\n환경 확인 완료!")

PyTorch    2.10.0
NumPy      2.4.2
Matplotlib 3.10.8
Apple Silicon GPU (MPS) 사용 가능

환경 확인 완료!


In [2]:
# ============================================================
# 공통 임포트 & 시드 설정
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)
print("임포트 완료, 시드 고정(42)")

임포트 완료, 시드 고정(42)


---
## 1. Positional Encoding: 순서 정보 추가 (ch4A 4.3)

Self-Attention은 **순서를 모른다**. "개가 사람을 물었다"와 "사람이 개를 물었다"를 구별하려면 위치 정보를 별도로 넣어줘야 한다.

두 가지 방식을 비교한다:
- **Sinusoidal** (원본 Transformer, 2017): 수학 함수로 고정 생성. 학습 파라미터 0개.
- **Learned** (BERT, GPT-2): 위치별 벡터를 학습. 파라미터 = max_len × d_model.

In [3]:
# ============================================================
# 1-1. Sinusoidal Positional Encoding
# ============================================================
class SinusoidalPositionalEncoding(nn.Module):
    """
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        """x: (batch, seq_len, d_model)"""
        return x + self.pe[:, :x.size(1), :]


# 1-2. Learned Positional Encoding
class LearnedPositionalEncoding(nn.Module):
    """BERT, GPT-2 등에서 사용하는 학습 가능한 PE"""
    def __init__(self, d_model, max_len=512):
        super().__init__()
        self.pe = nn.Embedding(max_len, d_model)

    def forward(self, x):
        seq_len = x.size(1)
        positions = torch.arange(seq_len, device=x.device)
        return x + self.pe(positions)


print("Positional Encoding 클래스 정의 완료")

Positional Encoding 클래스 정의 완료


In [4]:
# ============================================================
# 1-3. Sinusoidal vs Learned 비교
# ============================================================
d_model = 128
max_len = 100

sin_pe = SinusoidalPositionalEncoding(d_model, max_len)
learned_pe = LearnedPositionalEncoding(d_model, max_len)

sin_params = sum(p.numel() for p in sin_pe.parameters())
learned_params = sum(p.numel() for p in learned_pe.parameters())

print(f"[파라미터 수 비교]")
print(f"  Sinusoidal: {sin_params} (학습 파라미터 없음!)")
print(f"  Learned:    {learned_params:,} = {max_len} × {d_model}")

# PE 값 확인
pe_values = sin_pe.pe[0]  # (max_len, d_model)
print(f"\n[Sinusoidal PE — 첫 3개 위치, 처음 8차원]")
for pos in range(3):
    vals = [f"{pe_values[pos, d]:.3f}" for d in range(8)]
    print(f"  위치 {pos}: {vals}")

# 위치 간 코사인 유사도
print(f"\n[위치 간 코사인 유사도 — 가까울수록 1에 가까움]")
for target_pos in [1, 5, 10, 50]:
    sim = F.cosine_similarity(
        pe_values[0].unsqueeze(0), pe_values[target_pos].unsqueeze(0)
    )
    print(f"  위치 0 ↔ 위치 {target_pos:>2}: {sim.item():.4f}")

print(f"\n→ 가까운 위치일수록 유사도가 높다 = 상대적 거리가 자연스럽게 인코딩됨")

[파라미터 수 비교]
  Sinusoidal: 0 (학습 파라미터 없음!)
  Learned:    12,800 = 100 × 128

[Sinusoidal PE — 첫 3개 위치, 처음 8차원]
  위치 0: ['0.000', '1.000', '0.000', '1.000', '0.000', '1.000', '0.000', '1.000']
  위치 1: ['0.841', '0.540', '0.762', '0.648', '0.682', '0.732', '0.605', '0.796']
  위치 2: ['0.909', '-0.416', '0.987', '-0.160', '0.997', '0.071', '0.963', '0.269']

[위치 간 코사인 유사도 — 가까울수록 1에 가까움]
  위치 0 ↔ 위치  1: 0.9702
  위치 0 ↔ 위치  5: 0.7373
  위치 0 ↔ 위치 10: 0.6691
  위치 0 ↔ 위치 50: 0.5462

→ 가까운 위치일수록 유사도가 높다 = 상대적 거리가 자연스럽게 인코딩됨


---
## 2. Residual Connection & Layer Normalization (ch4A 4.4)

- **Residual**: `output = x + Sublayer(x)` — 원본 신호를 보존하는 "고속도로"
- **LayerNorm**: 벡터를 평균 0, 표준편차 1로 정규화 — "상대평가"

In [5]:
# ============================================================
# 2-1. Residual Connection 효과 실험
# ============================================================
x = torch.randn(1, 10, 128)  # (batch, seq, d_model)

# Residual 없이 20층 통과
signal_no_res = x.clone()
for i in range(20):
    layer = nn.Linear(128, 128)
    with torch.no_grad():
        signal_no_res = F.relu(layer(signal_no_res))

# Residual 있이 20층 통과
signal_with_res = x.clone()
for i in range(20):
    layer = nn.Linear(128, 128)
    with torch.no_grad():
        signal_with_res = signal_with_res + F.relu(layer(signal_with_res))

print(f"[Residual Connection 효과 — 20층 통과]")
print(f"  입력 신호 크기 (L2 norm): {x.norm().item():.4f}")
print(f"  Residual 없음: {signal_no_res.norm().item():.4f}  ← 신호 거의 사라짐!")
print(f"  Residual 있음: {signal_with_res.norm().item():.4f}  ← 신호 유지됨")
print(f"\n→ Residual 없이 깊은 네트워크는 작동 불가. 이것이 100층+ 모델을 가능하게 한다.")

[Residual Connection 효과 — 20층 통과]
  입력 신호 크기 (L2 norm): 35.6551
  Residual 없음: 1.4576  ← 신호 거의 사라짐!
  Residual 있음: 2057.6296  ← 신호 유지됨

→ Residual 없이 깊은 네트워크는 작동 불가. 이것이 100층+ 모델을 가능하게 한다.


In [6]:
# ============================================================
# 2-2. Layer Normalization 효과 확인
# ============================================================
x = torch.randn(2, 5, 128)  # (batch, seq, d_model)
layer_norm = nn.LayerNorm(128)
output = layer_norm(x)

print(f"[Layer Normalization 효과]")
print(f"  정규화 전 — 평균: {x[0, 0].mean().item():.4f}, 표준편차: {x[0, 0].std().item():.4f}")
print(f"  정규화 후 — 평균: {output[0, 0].mean().item():.4f}, 표준편차: {output[0, 0].std().item():.4f}")
print(f"\n→ LayerNorm은 각 위치에서 평균=0, 표준편차=1로 정규화한다 (상대평가!)")

[Layer Normalization 효과]
  정규화 전 — 평균: 0.0314, 표준편차: 1.1133
  정규화 후 — 평균: -0.0000, 표준편차: 1.0039

→ LayerNorm은 각 위치에서 평균=0, 표준편차=1로 정규화한다 (상대평가!)


---
## 3. Encoder Block 밑바닥 구현 (ch4A 4.4)

Encoder Block의 8단계:
1. 입력 → 2. 8개 헤드로 분할 → 3. 각 헤드가 Attention → 4. Concat →
5. Residual(1차) → 6. LayerNorm(1차) → 7. FFN → 8. Residual+LayerNorm(2차)

```
입력 ──┬── 8 Head Attention ── Concat ──(+)── LayerNorm ──┬── FFN ──(+)── LayerNorm ── 출력
       │                                  ↑               │                ↑
       └──────────── Residual ────────────┘               └── Residual ───┘
```

In [7]:
# ============================================================
# 3-1. Multi-Head Attention 구현
# ============================================================
class MultiHeadAttention(nn.Module):
    """
    MultiHead(Q, K, V) = Concat(head_1, ..., head_h) W^O
    head_i = Attention(QW_i^Q, KW_i^K, VW_i^V)
    """
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        # Linear projection
        Q = self.W_q(query)
        K = self.W_k(key)
        V = self.W_v(value)

        # Reshape: (batch, seq, d_model) → (batch, heads, seq, d_k)
        Q = Q.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        # Scaled Dot-Product Attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))

        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        context = torch.matmul(attention_weights, V)

        # Concat + Linear
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        output = self.W_o(context)
        return output, attention_weights


print("MultiHeadAttention 클래스 정의 완료")

MultiHeadAttention 클래스 정의 완료


In [8]:
# ============================================================
# 3-2. Position-wise Feed-Forward Network
# ============================================================
class PositionwiseFeedForward(nn.Module):
    """
    FFN(x) = GELU(xW₁ + b₁)W₂ + b₂

    d_model → d_ff(4배로 펼침) → d_model(다시 압축)
    활성화함수(GELU)가 "취사선택" 역할
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.dropout(F.gelu(self.linear1(x)))  # 펼침 + 취사선택
        x = self.linear2(x)                         # 압축
        return x


print("PositionwiseFeedForward 클래스 정의 완료")

PositionwiseFeedForward 클래스 정의 완료


In [9]:
# ============================================================
# 3-3. Encoder Block 조립
# ============================================================
class TransformerEncoderBlock(nn.Module):
    """
    Encoder Block (Post-LN 방식)
    Self-Attention → Add & Norm → FFN → Add & Norm
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Self-Attention + Residual + LayerNorm
        attn_output, attn_weights = self.self_attention(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))  # Residual!

        # FFN + Residual + LayerNorm
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))    # Residual!

        return x, attn_weights


print("TransformerEncoderBlock 클래스 정의 완료")

TransformerEncoderBlock 클래스 정의 완료


In [10]:
# ============================================================
# 3-4. Encoder Block 테스트
# ============================================================
d_model = 256
num_heads = 8
d_ff = 1024  # 4 × d_model
batch_size = 2
seq_len = 10

print(f"[모델 설정]")
print(f"  d_model: {d_model}")
print(f"  num_heads: {num_heads}")
print(f"  각 Head의 차원 (d_k): {d_model // num_heads}")
print(f"  d_ff: {d_ff}")

# Multi-Head Attention 테스트
mha = MultiHeadAttention(d_model, num_heads)
x = torch.randn(batch_size, seq_len, d_model)
output, attn_weights = mha(x, x, x)

print(f"\n[Multi-Head Attention]")
print(f"  입력 shape:  {x.shape}")
print(f"  출력 shape:  {output.shape}")
print(f"  Attention weights: {attn_weights.shape}")
print(f"    → (batch={batch_size}, heads={num_heads}, seq={seq_len}, seq={seq_len})")

# Encoder Block 테스트
encoder_block = TransformerEncoderBlock(d_model, num_heads, d_ff)
output_block, _ = encoder_block(x)

total_params = sum(p.numel() for p in encoder_block.parameters())
print(f"\n[Encoder Block]")
print(f"  입력 shape: {x.shape}")
print(f"  출력 shape: {output_block.shape}  ← 입출력 shape 동일!")
print(f"  파라미터 수: {total_params:,}")

print(f"\n  shape의 의미: torch.Size([{batch_size}, {seq_len}, {d_model}])")
print(f"    {batch_size} = batch (한 번에 처리하는 문장 수)")
print(f"    {seq_len} = seq_len (각 문장의 단어 수)")
print(f"    {d_model} = d_model (각 단어를 표현하는 벡터 차원)")

[모델 설정]
  d_model: 256
  num_heads: 8
  각 Head의 차원 (d_k): 32
  d_ff: 1024

[Multi-Head Attention]
  입력 shape:  torch.Size([2, 10, 256])
  출력 shape:  torch.Size([2, 10, 256])
  Attention weights: torch.Size([2, 8, 10, 10])
    → (batch=2, heads=8, seq=10, seq=10)

[Encoder Block]
  입력 shape: torch.Size([2, 10, 256])
  출력 shape: torch.Size([2, 10, 256])  ← 입출력 shape 동일!
  파라미터 수: 789,760

  shape의 의미: torch.Size([2, 10, 256])
    2 = batch (한 번에 처리하는 문장 수)
    10 = seq_len (각 문장의 단어 수)
    256 = d_model (각 단어를 표현하는 벡터 차원)


---
## 4. Decoder Block 구현 (ch4A 4.5)

Decoder는 Encoder보다 서브레이어가 하나 더 많다:
1. **Masked Self-Attention** — 미래 토큰을 보지 못하게 함 (Causal Masking)
2. **Cross-Attention** — Encoder 출력을 Q(Decoder), K·V(Encoder)로 참조
3. **FFN** — 정보 취사선택

예시: "나는 학교에 갔다" → "I went to school"

### 훈련 vs 추론
| | 훈련 (모델 만들기) | 추론 (모델 사용하기) |
|---|---|---|
| 정답 | 있음 | 없음 |
| 입력 | 정답 전체 한꺼번에 (Teacher Forcing) | 한 단어씩 순차 생성 (Autoregressive) |
| 병렬 | 가능 | 불가능 |

In [11]:
# ============================================================
# 4-1. Causal Mask 생성 및 확인
# ============================================================
def create_causal_mask(seq_len):
    """Causal (Look-ahead) Mask: 하삼각 행렬"""
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask.unsqueeze(0).unsqueeze(0)  # (1, 1, seq_len, seq_len)


# 5×5 예시로 확인
mask = create_causal_mask(5)
print("[Causal Mask 5×5]")
tokens = ["<SOS>", "I", "went", "to", "school"]
print(f"  토큰: {tokens}\n")

for i in range(5):
    row = [f"{mask[0, 0, i, j]:.0f}" for j in range(5)]
    visible = [tokens[j] for j in range(5) if mask[0, 0, i, j] == 1]
    print(f"  {tokens[i]:>8} 예측 시: [{', '.join(row)}]  → {visible}만 봄")

print(f"\n→ 각 위치에서 자신과 이전 토큰만 볼 수 있다 (미래는 가림)")
print(f"  이것이 Teacher Forcing에서 '커닝 방지' 역할을 한다")

[Causal Mask 5×5]
  토큰: ['<SOS>', 'I', 'went', 'to', 'school']

     <SOS> 예측 시: [1, 0, 0, 0, 0]  → ['<SOS>']만 봄
         I 예측 시: [1, 1, 0, 0, 0]  → ['<SOS>', 'I']만 봄
      went 예측 시: [1, 1, 1, 0, 0]  → ['<SOS>', 'I', 'went']만 봄
        to 예측 시: [1, 1, 1, 1, 0]  → ['<SOS>', 'I', 'went', 'to']만 봄
    school 예측 시: [1, 1, 1, 1, 1]  → ['<SOS>', 'I', 'went', 'to', 'school']만 봄

→ 각 위치에서 자신과 이전 토큰만 볼 수 있다 (미래는 가림)
  이것이 Teacher Forcing에서 '커닝 방지' 역할을 한다


In [12]:
# ============================================================
# 4-2. Decoder Block 구현
# ============================================================
class TransformerDecoderBlock(nn.Module):
    """
    Decoder Block:
    Masked Self-Attention → Add & Norm →
    Cross-Attention (Encoder 참조) → Add & Norm →
    FFN → Add & Norm
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.masked_self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        # 1. Masked Self-Attention (미래 가림)
        attn_output, self_attn_weights = self.masked_self_attention(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_output))

        # 2. Cross-Attention (Q: Decoder, K·V: Encoder)
        cross_output, cross_attn_weights = self.cross_attention(
            x, encoder_output, encoder_output, src_mask
        )
        x = self.norm2(x + self.dropout(cross_output))

        # 3. FFN
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))

        return x, self_attn_weights, cross_attn_weights


print("TransformerDecoderBlock 클래스 정의 완료")

TransformerDecoderBlock 클래스 정의 완료


In [13]:
# ============================================================
# 4-3. Decoder Block 테스트 + Causal Mask 검증
# ============================================================
# 먼저 Encoder로 입력 처리
encoder_block = TransformerEncoderBlock(d_model, num_heads, d_ff)
encoder_input = torch.randn(batch_size, seq_len, d_model)
encoder_output, _ = encoder_block(encoder_input)

# Decoder Block 실행
decoder_block = TransformerDecoderBlock(d_model, num_heads, d_ff)
decoder_input = torch.randn(batch_size, seq_len, d_model)
causal_mask = create_causal_mask(seq_len)

decoder_output, self_attn, cross_attn = decoder_block(
    decoder_input, encoder_output, tgt_mask=causal_mask
)

enc_params = sum(p.numel() for p in encoder_block.parameters())
dec_params = sum(p.numel() for p in decoder_block.parameters())

print(f"[Decoder Block 테스트]")
print(f"  Encoder 출력 shape: {encoder_output.shape}")
print(f"  Decoder 입력 shape: {decoder_input.shape}")
print(f"  Decoder 출력 shape: {decoder_output.shape}")
print(f"\n  Self-Attention weights:  {self_attn.shape}")
print(f"  Cross-Attention weights: {cross_attn.shape}")

print(f"\n[파라미터 수 비교]")
print(f"  Encoder Block: {enc_params:,}")
print(f"  Decoder Block: {dec_params:,}")
print(f"  → Decoder가 약 {(dec_params/enc_params - 1)*100:.0f}% 더 많음 (Cross-Attention 추가)")

# Causal Mask 적용 확인
print(f"\n[Causal Mask 동작 확인 — Self-Attention Weights]")
first_row = self_attn[0, 0, 0].detach()
second_row = self_attn[0, 0, 1].detach()
print(f"  첫 번째 토큰: [{first_row[0]:.3f}, {first_row[1]:.3f}, ..., {first_row[-1]:.3f}]")
print(f"    → 자기 자신만 참조 (나머지 ≈ 0)")
print(f"  두 번째 토큰: [{second_row[0]:.3f}, {second_row[1]:.3f}, {second_row[2]:.3f}, ...]")
print(f"    → 위치 0, 1만 참조 (위치 2부터 ≈ 0)")

[Decoder Block 테스트]
  Encoder 출력 shape: torch.Size([2, 10, 256])
  Decoder 입력 shape: torch.Size([2, 10, 256])
  Decoder 출력 shape: torch.Size([2, 10, 256])

  Self-Attention weights:  torch.Size([2, 8, 10, 10])
  Cross-Attention weights: torch.Size([2, 8, 10, 10])

[파라미터 수 비교]
  Encoder Block: 789,760
  Decoder Block: 1,053,440
  → Decoder가 약 33% 더 많음 (Cross-Attention 추가)

[Causal Mask 동작 확인 — Self-Attention Weights]
  첫 번째 토큰: [1.111, 0.000, ..., 0.000]
    → 자기 자신만 참조 (나머지 ≈ 0)
  두 번째 토큰: [0.618, 0.493, 0.000, ...]
    → 위치 0, 1만 참조 (위치 2부터 ≈ 0)


---
## 5. 전체 Transformer Encoder 스택 + 분류 모델

동일한 Encoder Block을 N번 쌓으면 Transformer Encoder가 완성된다.
여기에 Token Embedding + Positional Encoding + 분류 Head를 붙이면
텍스트 분류 모델이 된다 (BERT의 구조와 동일).

In [14]:
# ============================================================
# 5-1. N-Layer Encoder + 분류 모델
# ============================================================
class TransformerEncoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

    def forward(self, x, mask=None):
        attn_weights_all = []
        for layer in self.layers:
            x, attn_weights = layer(x, mask)
            attn_weights_all.append(attn_weights)
        return x, attn_weights_all


class TransformerClassifier(nn.Module):
    """Transformer Encoder 기반 텍스트 분류 (BERT 구조)"""
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers,
                 num_classes, max_len=512, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = SinusoidalPositionalEncoding(d_model, max_len)
        self.encoder = TransformerEncoder(d_model, num_heads, d_ff, num_layers, dropout)
        self.classifier = nn.Linear(d_model, num_classes)
        self.dropout = nn.Dropout(dropout)
        self.d_model = d_model

    def forward(self, x, mask=None):
        x = self.embedding(x) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        x = self.dropout(x)
        x, _ = self.encoder(x, mask)
        x = x.mean(dim=1)  # Mean Pooling
        return self.classifier(x)


print("TransformerEncoder, TransformerClassifier 클래스 정의 완료")

TransformerEncoder, TransformerClassifier 클래스 정의 완료


In [ ]:
# ============================================================
# 5-2. 전체 모델 테스트
# ============================================================
num_layers = 4
vocab_size = 10000
num_classes = 2

# 4-Layer Encoder 스택
encoder = TransformerEncoder(d_model, num_heads, d_ff, num_layers)
x = torch.randn(batch_size, seq_len, d_model)
output_enc, _ = encoder(x)

enc_total = sum(p.numel() for p in encoder.parameters())
print(f"[{num_layers}층 Transformer Encoder]")
print(f"  입력: {x.shape}")
print(f"  출력: {output_enc.shape}")
print(f"  총 파라미터: {enc_total:,}개")

# 분류 모델
classifier = TransformerClassifier(
    vocab_size, d_model, num_heads, d_ff, num_layers, num_classes
)
input_ids = torch.randint(0, vocab_size, (batch_size, seq_len))
logits = classifier(input_ids)

cls_total = sum(p.numel() for p in classifier.parameters())
print(f"\n[Transformer Classifier]")
print(f"  vocab_size: {vocab_size}")
print(f"  num_classes: {num_classes}")
print(f"  입력: {input_ids.shape} (정수 토큰 ID)")
print(f"  출력: {logits.shape} (클래스별 점수)")
print(f"  총 파라미터: {cls_total:,}개")

In [ ]:
# ============================================================
# 5-3. 실제 모델과 크기 비교
# ============================================================
print(f"[모델 크기 비교]")
print(f"""  
  | 모델              | d_model | heads | layers | d_ff   | 파라미터        |
  |-------------------|---------|-------|--------|--------|----------------|
  | 우리 모델         | {d_model:>5}   | {num_heads:>3}   | {num_layers:>4}   | {d_ff:>4}   | {cls_total:>12,}  |
  | BERT-base         |   768   |  12   |   12   | 3,072  |        ~110M   |
  | GPT-2 small       |   768   |  12   |   12   | 3,072  |        ~117M   |
  | GPT-2 XL          | 1,600   |  25   |   48   | 6,400  |      ~1,500M   |
  | Llama 3 70B       | 8,192   |  64   |   80   | 28,672 |     ~70,000M   |
  | Llama 3 405B      | 16,384  | 128   |  126   | 53,248 |    ~405,000M   |
""")
print(f"  깊이(layers) = 사고의 반복 횟수")
print(f"  너비(d_model) = 사고의 해상도")
print(f"  현대 LLM은 둘 다 키운 것이다.")

In [15]:
# ============================================================
# 5-4. PyTorch 내장 모듈과 비교
# ============================================================
pytorch_layer = nn.TransformerEncoderLayer(
    d_model=d_model, nhead=num_heads, dim_feedforward=d_ff,
    dropout=0.1, batch_first=True
)
pytorch_encoder = nn.TransformerEncoder(pytorch_layer, num_layers=num_layers)

x = torch.randn(batch_size, seq_len, d_model)
output_pytorch = pytorch_encoder(x)

pytorch_params = sum(p.numel() for p in pytorch_encoder.parameters())
custom_params = sum(p.numel() for p in encoder.parameters())

print(f"[PyTorch 내장 vs 직접 구현 비교]")
print(f"  PyTorch 출력 shape: {output_pytorch.shape}")
print(f"  PyTorch 파라미터:   {pytorch_params:,}")
print(f"  직접 구현 파라미터: {custom_params:,}")
print(f"\n→ 구조가 동일하므로 파라미터 수도 거의 같다!")
print(f"  실제 프로젝트에서는 PyTorch 내장 모듈을 쓰면 된다.")
print(f"  밑바닥 구현은 '원리 이해'를 위한 것이다.")

NameError: name 'num_layers' is not defined

---
## 실습 완료!

이 노트북에서 구현한 것들:

1. **Positional Encoding** — Sinusoidal(고정) vs Learned(학습) 비교
2. **Residual Connection** — 20층 통과 실험으로 효과 확인
3. **Layer Normalization** — 평균 0, 표준편차 1로 정규화 확인
4. **Multi-Head Attention** — 8개 헤드가 각자 다른 관점으로 분석
5. **FFN** — 확장(펼침) → 활성화(취사선택) → 축소(압축)
6. **Encoder Block** — MHA + Residual + LN + FFN + Residual + LN
7. **Decoder Block** — Masked Self-Attention + Cross-Attention + FFN
8. **Causal Mask** — 미래 토큰을 가리는 하삼각 행렬
9. **Transformer Classifier** — Encoder 스택 + 분류 Head (BERT 구조)

다음 주(5장)에서는 이 구조 위에 사전 학습(Pre-training)을 올려 BERT와 GPT를 구현합니다.